# 04 Neurosymbolic Inference Heatmap

Notebook demo final untuk inference Neuro-Symbolic dari gambar upload.

- Faster R-CNN tetap menyediakan backbone, RPN, RoI Align, bbox regressor, dan post-processing deteksi.
- SODT menggantikan cabang klasifikasi dan menerima flattened RoI Align pooled grid `[C, 7, 7]`.
- Explanation yang ditampilkan adalah **Local Evidence per decision node**, bukan heatmap path gabungan.


In [1]:
from pathlib import Path
import io

import ipywidgets as widgets
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import clear_output, display
from PIL import Image

from notebooks.util import resolve_root
from neuro.config import NeuroConfig, NeuroTrainConfig
from neuro.preprocess_dataset import test_preprocess
from neurosym.inference import (
    explain_hybrid_detection,
    load_neurosymbolic_detector,
    run_neurosymbolic_inference,
    select_detection_indices,
)
from util.artifacts import latest_run_checkpoint
from util.config import load_yaml
from util.device import select_device

PROJECT_ROOT = resolve_root()

In [2]:
neuro_config = load_yaml(Path("neuro.yaml"), NeuroConfig)
train_config = load_yaml(Path("neuro_train.yaml"), NeuroTrainConfig)

device = select_device(train_config["device"])
detector_checkpoint_path = latest_run_checkpoint(PROJECT_ROOT / "checkpoints" / "neuro")
symbolic_dir = PROJECT_ROOT / "checkpoints" / "symbolic"
symbolic_checkpoint_path = symbolic_dir / "run1.pt"

hybrid_model, detector_checkpoint = load_neurosymbolic_detector(
    detector_checkpoint_path=detector_checkpoint_path,
    neuro_config=neuro_config,
    train_config=train_config,
    symbolic_checkpoint_path=symbolic_checkpoint_path,
    device=str(device),
)

image_preprocess = test_preprocess()
class_names = tuple(train_config["dataset"]["class_names"])

detector_checkpoint_path, symbolic_checkpoint_path

(PosixPath('/home/wiszel/OneDrive/Documents/WiszeL/UNS/Semester 8/pcb_skripsi/checkpoints/neuro/run2.pt'),
 PosixPath('/home/wiszel/OneDrive/Documents/WiszeL/UNS/Semester 8/pcb_skripsi/checkpoints/symbolic/run1.pt'))

In [3]:
def image_to_array(image_tensor: torch.Tensor) -> np.ndarray:
    return image_tensor.detach().cpu().permute(1, 2, 0).clamp(0.0, 1.0).numpy()


def heatmap_to_array(heatmap: torch.Tensor) -> np.ndarray:
    return heatmap.detach().cpu().numpy()


def class_name(label: int) -> str:
    return class_names[int(label) - 1]


def zoom_axis_to_box(
    axis: plt.Axes,
    box: torch.Tensor,
    image_shape: tuple[int, int],
    padding_ratio: float = 0.85,
    minimum_crop_size: int = 96,
) -> None:
    image_height, image_width = int(image_shape[0]), int(image_shape[1])
    x1, y1, x2, y2 = [float(value) for value in box.detach().cpu().tolist()]
    box_width = max(x2 - x1, 1.0)
    box_height = max(y2 - y1, 1.0)
    crop_width = max(box_width * (1.0 + 2.0 * padding_ratio), float(minimum_crop_size))
    crop_height = max(
        box_height * (1.0 + 2.0 * padding_ratio), float(minimum_crop_size)
    )
    center_x = (x1 + x2) / 2.0
    center_y = (y1 + y2) / 2.0

    left = max(center_x - crop_width / 2.0, 0.0)
    right = min(center_x + crop_width / 2.0, float(image_width))
    top = max(center_y - crop_height / 2.0, 0.0)
    bottom = min(center_y + crop_height / 2.0, float(image_height))

    axis.set_xlim(left, right)
    axis.set_ylim(bottom, top)


def draw_numbered_detections(
    axis: plt.Axes,
    image_tensor: torch.Tensor,
    detection_result: dict[str, torch.Tensor],
    detection_indices: list[int],
    selected_index: int | None = None,
    display_numbers: list[int] | None = None,
) -> None:
    axis.imshow(image_to_array(image_tensor))
    axis.axis("off")

    if display_numbers is None:
        display_numbers = list(range(1, len(detection_indices) + 1))

    for display_number, detection_index in zip(display_numbers, detection_indices):
        box = detection_result["boxes"][detection_index].detach().cpu()
        label = int(detection_result["labels"][detection_index])
        score = float(detection_result["scores"][detection_index])
        x1, y1, x2, y2 = box.tolist()
        edge_color = "lime" if detection_index == selected_index else "red"
        line_width = 3 if detection_index == selected_index else 2
        axis.add_patch(
            patches.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=line_width,
                edgecolor=edge_color,
                facecolor="none",
                clip_on=True,
            )
        )
        axis.text(
            x1,
            max(y1 - 5, 0),
            f"#{display_number} {class_name(label)} {score:.2f}",
            color="black",
            fontsize=9,
            weight="bold",
            clip_on=True,
            bbox={"facecolor": "yellow", "edgecolor": edge_color, "pad": 2},
        )

In [4]:
upload_widget = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload Image",
)
MAX_DISPLAY_DETECTIONS = 9
DISPLAY_SCORE_THRESHOLD = 0.3
detection_output = widgets.Output()
explanation_output = widgets.Output()
state = {
    "image_name": None,
    "image_tensor": None,
    "detection": None,
    "selected_indices": [],
}


def uploaded_file_record():
    value = upload_widget.value
    if isinstance(value, tuple):
        return value[0] if value else None
    if isinstance(value, dict):
        if "content" in value:
            return value
        return next(iter(value.values())) if value else None
    return None


def uploaded_content_bytes(file_record) -> bytes:
    content = file_record["content"]
    return content.tobytes() if isinstance(content, memoryview) else bytes(content)


def render_detection(detection_index: int) -> None:
    image_tensor = state["image_tensor"]
    detection = state["detection"]
    selected_indices = state["selected_indices"]
    explanation = explain_hybrid_detection(
        hybrid_model,
        detection,
        detection_index=detection_index,
        image_shape=tuple(image_tensor.shape[-2:]),
    )
    label_name = class_name(explanation["label"])

    with explanation_output:
        clear_output(wait=True)
        fig, axis = plt.subplots(figsize=(8, 8))
        selected_number = selected_indices.index(detection_index) + 1
        draw_numbered_detections(
            axis,
            image_tensor,
            detection,
            [detection_index],
            selected_index=detection_index,
            display_numbers=[selected_number],
        )
        zoom_axis_to_box(
            axis, explanation["detection_box"], tuple(image_tensor.shape[-2:])
        )
        axis.set_title(
            f"Zoomed detection #{selected_number}: {label_name} {explanation['score']:.2f}"
        )
        plt.show()

        node_rows = [
            {
                "step": node["depth"] + 1,
                "node_index": node["node_index"],
                "decision": node["decision"],
                "score": node["score"],
                "active_roi_units": node["active_original_feature_count"],
            }
            for node in explanation["node_explanations"]
        ]
        display(pd.DataFrame(node_rows))

        node_count = len(explanation["node_explanations"])
        columns = min(3, node_count)
        rows = (node_count + columns - 1) // columns
        fig, axes = plt.subplots(
            rows, columns, figsize=(5 * columns, 5 * rows), squeeze=False
        )
        axes = axes.reshape(-1)

        for axis_index, node in enumerate(explanation["node_explanations"]):
            axis = axes[axis_index]
            axis.imshow(image_to_array(image_tensor))
            axis.imshow(
                heatmap_to_array(node["projected_node_heatmap"]),
                cmap="inferno",
                alpha=0.65,
                vmin=0.0,
                vmax=1.0,
            )
            x1, y1, x2, y2 = explanation["detection_box"].tolist()
            axis.add_patch(
                patches.Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    linewidth=2,
                    edgecolor="cyan",
                    facecolor="none",
                )
            )
            zoom_axis_to_box(
                axis, explanation["detection_box"], tuple(image_tensor.shape[-2:])
            )
            axis.set_title(
                f"Step {node['depth'] + 1} | Node {node['node_index']} -> {node['decision']}"
            )
            axis.axis("off")

        for axis in axes[node_count:]:
            axis.axis("off")

        plt.tight_layout()
        plt.show()


def make_detection_button(display_number: int, detection_index: int) -> widgets.Button:
    detection = state["detection"]
    label = int(detection["labels"][detection_index])
    score = float(detection["scores"][detection_index])
    button = widgets.Button(
        description=f"#{display_number} {class_name(label)} {score:.2f}",
        layout=widgets.Layout(width="180px"),
    )
    button.on_click(lambda _: render_detection(detection_index))
    return button


def run_uploaded_inference(_button) -> None:
    file_record = uploaded_file_record()
    with detection_output:
        clear_output(wait=True)
        explanation_output.clear_output(wait=True)

        if file_record is None:
            print("Upload one PCB image first.")
            return

        image_name = file_record.get("name", "uploaded_image")
        pil_image = Image.open(io.BytesIO(uploaded_content_bytes(file_record))).convert(
            "RGB"
        )
        image_tensor = image_preprocess(pil_image)
        detection = run_neurosymbolic_inference(hybrid_model, [image_tensor])[0]
        selected_indices = select_detection_indices(
            detection,
            score_threshold=DISPLAY_SCORE_THRESHOLD,
            max_detections=MAX_DISPLAY_DETECTIONS,
        )

        state["image_name"] = image_name
        state["image_tensor"] = image_tensor
        state["detection"] = detection
        state["selected_indices"] = selected_indices

        fig, axis = plt.subplots(figsize=(8, 8))
        draw_numbered_detections(axis, image_tensor, detection, selected_indices)
        axis.set_title(f"Neuro-Symbolic detections: {image_name}")
        plt.show()

        if not selected_indices:
            print("No detections were returned by the model.")
            return

        buttons = [
            make_detection_button(display_number, detection_index)
            for display_number, detection_index in enumerate(selected_indices, start=1)
        ]
        display(
            widgets.GridBox(
                buttons,
                layout=widgets.Layout(
                    grid_template_columns="repeat(3, 190px)",
                    grid_gap="8px",
                ),
            )
        )

    render_detection(selected_indices[0])


def run_uploaded_inference_on_upload(change) -> None:
    if change["new"]:
        run_uploaded_inference(None)


upload_widget.observe(run_uploaded_inference_on_upload, names="value")
display(
    widgets.VBox(
        [
            upload_widget,
            detection_output,
            explanation_output,
        ]
    )
)